In [1]:
from dataclasses import dataclass

In [35]:
from dataclasses import dataclass

@dataclass
class Config:
    embed_dim: int
    hidden_dim: int
    dtype: str
    seq_length: int

    num_heads: int
    rope_base_theta: int 

In [4]:
from dataclasses import dataclass
import torch
import torch.nn as nn


class FeedForward(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.fc1 = nn.Linear(cfg.embed_dim, cfg.hidden_dim, dtype=cfg.dtype, bias=False)
        self.fc2 = nn.Linear(cfg.embed_dim, cfg.hidden_dim, dtype=cfg.dtype, bias=False)
        self.fc3 = nn.Linear(cfg.hidden_dim, cfg.embed_dim, dtype=cfg.dtype, bias=False)

    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.gelu(x_fc1, approximate="tanh") * x_fc2
        return self.fc3(x)

class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.zeros(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    def forward(self, x):
        # Match HF Gemma3: compute norm in float32, then scale by (1 + w)
        input_dtype = x.dtype
        x_f = x.float()
        var = x_f.pow(2).mean(dim=-1, keepdim=True)
        x_norm = x_f * torch.rsqrt(var + self.eps)
        out = x_norm * (1.0 + self.scale.float())
         
        if self.shift is not None:
            out = out + self.shift.float()
         
        return out.to(input_dtype)

In [ ]:
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False,
        query_pre_attn_scalar=None, dtype=None, ):
        super().__init__()



    def forward(self, x):
        pass

    def _precompute_rotary_embeddings(self, seq_len, head_dim, base=100000, device=None, dtype: torch.dtype = torch.float32):
        channel_range = torch.arange(0, head_dim, 2, dtype=torch.float32)
        inv_freq = 1.0 / (base ** (channel_range / head_dim))

        t = torch.arange(seq_len, dtype=torch.float32, device=device)

        freqs = torch.outer(t, inv_freq)

        cos, sin = freqs.cos(), freqs.sin()
        cos, sin = cos.to(dtype), sin.to(dtype)
        cos, sin = cos[None, :, None, :], sin[None, :, None, :] 
        return cos, sin


    def apply_rotary_embs(self, x, cos, sin):
        d = x.shape[-1] // 2
        x1, x2 = x[..., :d], x[..., d:] 

        y1 = x1 * cos + x2 * sin 
        y2 = x1 * (-sin) + x2 * cos
        return torch.cat([y1, y2], 3)


In [ ]:

    

class Goda(nn.Module):
    def __init__(self, config: Config) -> None:
        super().__init__()
        self.config = config

In [17]:
cos, sin = Goda(None)._precompute_rotary_embeddings(32, 8)

In [19]:
cos.shape, sin.shape

(torch.Size([1, 32, 1, 4]), torch.Size([1, 32, 1, 4]))

In [31]:
x= torch.rand([1, 32, 6, 8])

In [32]:
y = Goda(None).apply_rotary_embs(x, cos, sin)

In [33]:
y.shape

torch.Size([1, 32, 6, 8])